# Regenerate the `llm_yesno` feature with the fine-tuned judge

**Run-all safe. Read-only. No footguns.**
- **Reads** the trained adapter from `ADAPTER_DIR` — never trains, never writes to it.
- **Reuses the exact `(source, topic_id, doc_id)` pairs** from the existing zero-shot feature file
  (`llm_reranker_scores.jsonl`), so the ensemble comparison is a *clean feature swap* — same pool,
  same prompt, only the judge changes. No BM25/dense rebuild needed.
- **Writes only** `llm_scores_ft.jsonl` (same schema `{source, topic_id, doc_id, llm_score}`).
  Resumable per pair — if the runtime disconnects, just run-all again and it continues.
- Scores TREC21 + KZ + TREC22. Scoring TREC22 here is *test-time feature inference* (the judge
  never sees labels) — it is not the one-shot. The one-shot is when you *evaluate the ensemble* on
  TREC22, which stays gated in `train_ensemble_full.ipynb` (see the A/B cell there).

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers peft accelerate datasets tqdm
# peft hard-requires torchao>=0.16.0 inside PeftModel.from_pretrained; Colab often ships 0.10.0,
# which raises ImportError. Upgrade LAST so nothing re-resolves it down, then RESTART the runtime.
!pip install -q -U "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT       = '/content/drive/MyDrive/ct_data23'
EVAL_ROOT       = f'{DATA_ROOT}/evaluation'
TREC_ROOT       = f'{EVAL_ROOT}/trec_data'
KZ_ROOT         = f'{EVAL_ROOT}/kz_data'
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
ADAPTER_DIR     = f'{DATA_ROOT}/judge_lora_v1'                 # trained adapter (READ ONLY)
BASE_MODEL      = 'Qwen/Qwen2.5-7B-Instruct'
ZS_SCORES       = f'{DATA_ROOT}/llm_reranker_scores.jsonl'     # source of the exact pairs to re-score
FT_OUT          = f'{DATA_ROOT}/llm_scores_ft.jsonl'          # the only file this notebook writes

REGEN_SETS      = ['trec21', 'kz', 'trec22']   # complete drop-in; TREC22 is feature inference, not eval
DOC_CHARS       = 1800     # must match the fine-tune / zero-shot prompt
MAX_LEN         = 1024     # matches the length the judge was trained at
BATCH           = 16
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
assert os.path.isdir(ADAPTER_DIR), f'adapter not found at {ADAPTER_DIR}'
assert os.path.exists(ZS_SCORES), f'zero-shot pairs file not found at {ZS_SCORES}'
print('config set | regen sets:', REGEN_SETS)

In [ ]:
import json
from collections import defaultdict
from datasets import load_dataset
from ctmatch.evaluation.eval_utils import load_eval_datasets

# corpus text (trial docs)
_idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in _idx]
with open(FULLTEXT_CORPUS) as f:
    corpus_txt = [l.rstrip('\n') for l in f]
id2txt = dict(zip(corpus_ids, corpus_txt))

# patient text (topics)
all_sets = load_eval_datasets(TREC_ROOT, KZ_ROOT)
all_sets = {k: v for k, v in all_sets.items() if k in REGEN_SETS}

# exact pairs to re-score, taken from the zero-shot feature file (guarantees a clean A/B)
pairs = defaultdict(list)
with open(ZS_SCORES) as f:
    for line in f:
        r = json.loads(line)
        if r['source'] in REGEN_SETS:
            pairs[(r['source'], r['topic_id'])].append(r['doc_id'])
print(f'topics to score: {len(pairs)} | pairs: {sum(len(v) for v in pairs.values()):,}')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

dev = 'cuda'
tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16,
                                            attn_implementation='sdpa').to(dev)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(dev)   # epoch-2 best adapter (DoRA-aware, read-only)
model.config.use_cache = True; model.eval()

YES = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['yes','Yes',' yes',' Yes','YES']})
NO  = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['no','No',' no',' No','NO']})
SYS = 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'
def make_prompt(patient, trial):
    u = (f'You are a clinical trial matching expert.\n\nPatient:\n{patient}\n\n'
         f'Trial:\n{trial[:DOC_CHARS]}\n\nIs this patient likely eligible for this trial? '
         'Answer with a single word: yes or no.')
    return tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':u}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def judge_scores(patient, trials):
    out = []
    for i in range(0, len(trials), BATCH):
        prompts = [make_prompt(patient, t) for t in trials[i:i+BATCH]]
        e = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(dev)
        last = model(**e).logits[:, -1, :].float()
        out.extend((torch.logsumexp(last[:, YES], -1) - torch.logsumexp(last[:, NO], -1)).cpu().tolist())
    return out
print('fine-tuned judge loaded (read-only) from', ADAPTER_DIR)

In [ ]:
from tqdm.auto import tqdm

# resume: skip (source, topic, doc) already written
done = set()
if os.path.exists(FT_OUT):
    with open(FT_OUT) as f:
        for line in f:
            r = json.loads(line); done.add((r['source'], r['topic_id'], r['doc_id']))
print(f'already scored: {len(done):,} pairs')

n_new = 0
with open(FT_OUT, 'a') as out_f:
    for (src, tid), docs in tqdm(pairs.items(), desc='FT judge'):
        patient = all_sets[src]['topic2text'].get(tid)
        if patient is None: continue
        todo = [d for d in docs if (src, tid, d) not in done and d in id2txt]
        if not todo: continue
        scores = judge_scores(patient, [id2txt[d] for d in todo])
        for d, s in zip(todo, scores):
            out_f.write(json.dumps({'source': src, 'topic_id': tid, 'doc_id': d,
                                    'llm_score': float(s)}) + '\n')
        out_f.flush(); n_new += len(todo)

total = sum(1 for _ in open(FT_OUT))
print(f'\nadded {n_new:,} new scores | total {total:,} -> {FT_OUT}')
print('NEXT: open train_ensemble_full.ipynb, run all, read the A/B cell (TREC21 CV: zero-shot vs fine-tuned).')